# Week 3 — Data Contract on the Full Warehouse Release

**Author:** Zeyad
**Date:** 2026-07-19
**Lane:** 2 — Refresh / Content Opportunity Scoring (per `w01_research_question.ipynb`,
`w02_ml_task_framing.ipynb`)

This is the first notebook that runs against the real warehouse (`FlyRank/internship-warehouse`
on Hugging Face) instead of the 30k-row starter CSV. The job is small on purpose: write the
contract in plain words, prove three facts about my slice with real queries, build five features,
then reproduce the leakage lesson from notebook 02 on real warehouse data instead of the starter
slice.

Per the assignment's own warning: `fact_content_daily_performance_sample` is the sealed final
month (June 2026) — test-only, never for label logic. Everything below iterates on a mid-panel
month, `month=2026-03`, and treats the final month as untouched.

## 0. Setup

Same pattern as `notebooks/03_working_with_the_full_release.ipynb`: DuckDB talks to the
Hugging Face release directly, no local download of the full table. I'm pointing at the
`month=2026-03` partition specifically rather than the whole 79M-row fact table — that's the
whole point of the partition, and it keeps this notebook fast and rate-limit-friendly.

In [1]:
%pip install -q duckdb huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass

# Never hardcode the token in a cell -- this repo is public.
# Priority: Colab's Secrets panel (key icon, left sidebar) -> env var (CI) -> getpass prompt (local/other).
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":     f"read_parquet('{REL}/dim_clients.parquet')",
    # Partition path, not the full fact table -- this is the "iterate on a mid-panel month"
    # instruction taken literally: only March's parquet files get pulled over the network.
    "fact_march":       f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:12} {n:>12,} rows")

dim_clients           104 rows
fact_march      9,841,378 rows


**Why I'm not touching `dim_content` or `fact_content_query_90d` here, before I even get to the
contract:** the lane guide documents `dim_clients` and `fact_content_daily_performance` column by
column (`gsc_data_start`, `ga4_data_start`, `ga4_data_available`, `gsc_impressions`,
`gsc_clicks`, `gsc_avg_position`, the join keys). It never gives me the full `dim_content` column
list, and I don't have a way to browse the dataset's own schema page from here. Rather than
guessing column names and having the notebook break on a typo, I'm confirming the columns I
actually use with a schema peek below, and I'm leaving `dim_content` out of this week's contract
entirely — that's the limitation I name in section 4, not something to paper over.

`fact_content_query_90d` is a harder no, not just an unconfirmed-schema maybe: the data
dictionary's leakage warning says that table is a **fixed** 90-day window anchored to the export
date (`2026-06-23`), not a per-month table. For a March decision point that window doesn't even
line up with March — it sits close to the sealed final month. Joining it here would silently
staple future information onto a March row. That's my one deliberate exclusion in section 2.

In [4]:
# Confirm the columns I'm about to rely on actually exist under these names, instead of assuming.
con.sql(f"SELECT * FROM {TABLES['fact_march']} LIMIT 3").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. The contract, part A — row, tables, time window

**What one row means for my lane.** One row is one pseudonymized content page
(`content_hash_id`, within one `client_hash_id`), described by a decision-moment snapshot built
by aggregating that page's daily search performance up to a cutoff date. It is **not** one row
per day — the raw fact table's grain is daily, and I aggregate on top of it. That distinction is
exactly what fact A in section 3 exists to prove: if I skipped verifying the raw grain first, an
aggregation bug (e.g. an accidental fan-out on a bad join) could quietly duplicate a page's rows
and I'd never notice from the aggregate alone.

**Which table(s).** `dim_clients` (for per-client history coverage — `gsc_data_start` /
`ga4_data_start` — and for client-holdout grouping later) and `fact_content_daily_performance`,
restricted to the `month=2026-03` partition. Both have documented, confirmed columns. See the
setup section above for why `dim_content` and `fact_content_query_90d` are out for this week.

**Which time window.** For this notebook I'm using March 15, 2026 as a mid-month decision point,
splitting the `month=2026-03` partition into a **feature window** (days 1–15, "what I'd know by
the decision moment") and a **forward window** (days 16–31, "what happens next"). This is a
compressed, single-month stand-in for the real design I flagged at the end of `w02`: trailing
90 days of features predicting a next-30-days decline. I'm not building that full version yet —
the assignment is scoped to a single mid-panel month to test query mechanics honestly before I
spend real HF bandwidth on the full multi-month version.

## 2. The contract, part B — label/proxy and exclusion

**What I'd predict or rank.** Same lane target as `w02`: an urgency score used to rank pages for
refresh review. For this notebook's teaching-scale exercise, the proxy label is
`is_declining = 1` when a page's forward-window (days 16–31) impressions drop more than 20%
versus its feature-window (days 1–15) impressions, mirroring the starter CSV's
`trend_direction == "down"` rule but computed fresh from the raw daily warehouse table instead of
inherited from a pre-built column. This is still a same-window-adjacent proxy, not the real
future-looking capstone label — see the self-check for what's still open.

**One thing I deliberately exclude.** `fact_content_query_90d` — reasoned through in the setup
section above. It's a fixed window anchored near the export date, not a per-month table, so it
cannot honestly describe March without leaking information from months that, relative to a March
decision point, haven't happened yet.

## 3. Prove it — three queries, five features, the trap

### 3a. Fact one — the grain

If the raw fact table really is `report_date × client_hash_id × content_hash_id`, grouping by
that triple and counting rows should equal the row count of the whole slice. No mismatch, no
silent duplication.

In [5]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*)                                                      AS raw_rows,
        COUNT(*) FILTER (WHERE key_count = 1)                         AS rows_with_unique_key
    FROM (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS key_count
        FROM {TABLES['fact_march']}
        GROUP BY 1, 2, 3
    )
""").df()

print(grain_check)
assert grain_check.loc[0, "raw_rows"] == grain_check.loc[0, "rows_with_unique_key"], \
    "grain assumption broken -- (report_date, client_hash_id, content_hash_id) is not unique"
print("Confirmed: one row per (report_date, client_hash_id, content_hash_id) in the March slice.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   raw_rows  rows_with_unique_key
0   9841378               9841378
Confirmed: one row per (report_date, client_hash_id, content_hash_id) in the March slice.


### 3b. Fact two — slice size and date span

How big is my working slice, and does it actually cover the calendar month I asked for?

In [6]:
slice_shape = con.sql(f"""
    SELECT
        COUNT(*)                       AS row_count,
        MIN(report_date)               AS min_date,
        MAX(report_date)               AS max_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM {TABLES['fact_march']}
""").df()

slice_shape

,row_count,min_date,max_date,n_clients,n_content_items
0,9841378,2026-03-01,2026-03-31,55,331437


### 3c. Fact three — availability, filtered with `IS TRUE`

`ga4_data_available` is `FALSE` for rows before a client's GA4 start — zero-filled GA4 columns on
those rows mean "not measured," not "measured as zero." How many of March's rows actually carry
usable GA4 coverage?

In [7]:
availability = con.sql(f"""
    SELECT
        COUNT(*)                                      AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_march']}
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966,4.2


### 3d. Five features, decision point = March 15

Every feature below is built **only** from `report_date BETWEEN '2026-03-01' AND '2026-03-15'` —
the feature window, strictly before the forward window the label comes from in 3e.

- `imp_h1` (SUM of `gsc_impressions`, days 1–15) — knowable at the decision moment because it's
  the running impression total already logged by March 15; nothing about it depends on what
  happens March 16 onward.
- `clicks_h1` (SUM of `gsc_clicks`, days 1–15) — same reasoning, observed clicks through the
  cutoff.
- `avg_position_h1` (AVG of `gsc_avg_position`, days 1–15, excluding no-data rows) — the page's
  average observed ranking position through the cutoff; a snapshot of where it already stood.
- `active_days_h1` (COUNT of distinct days with `gsc_impressions > 0`, days 1–15) — how
  consistently the page showed up in search through the cutoff, not just its total volume.
- `ga4_covered_days_h1` (COUNT of distinct days with `ga4_data_available IS TRUE`, days 1–15) —
  an ingestion-time coverage flag, not an outcome; it's set by when the client's GA4 connection
  started, which is already known by the cutoff regardless of what the page does later.

In [8]:
features_h1 = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                              AS imp_h1,
        SUM(gsc_clicks)                                                   AS clicks_h1,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0)         AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)    AS active_days_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_covered_days_h1
    FROM {TABLES['fact_march']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
""").df()

print(f"{len(features_h1):,} (client, content) rows in the feature frame")
features_h1.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

319,759 (client, content) rows in the feature frame


,client_hash_id,content_hash_id,imp_h1,clicks_h1,avg_position_h1,active_days_h1,ga4_covered_days_h1
0,client_62f4a7e64f5e0096,content_c6bbcab7cd78c380,891.0,0.0,9.772905,15,0
1,client_62f4a7e64f5e0096,content_6d978c91fcf0c680,279.0,1.0,7.709344,15,0
2,client_62f4a7e64f5e0096,content_89f06671812ca9c7,304.0,0.0,5.944541,15,0
3,client_62f4a7e64f5e0096,content_d54c0e0ff7061da3,205.0,0.0,7.929196,15,0
4,client_62f4a7e64f5e0096,content_3fa077d5341ea420,119.0,1.0,9.612619,14,0


### 3e. The trap — one label-derived column, on purpose

The label: did the page's forward-window (days 16–31) impressions drop more than 20% versus its
feature-window (days 1–15) impressions? This is the exact same rule the starter CSV's
`trend_direction == "down"` uses (`docs/data-dictionary.md`), just computed fresh here instead of
inherited pre-built.

I'll build the honest features (`features_h1`, already strictly before the label window), then
deliberately add `imp_h2` — forward-window impressions, the literal quantity the label is
computed from — as an extra "feature," exactly the leak notebook 02 warned about.

In [9]:
forward = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_h2
    FROM {TABLES['fact_march']}
    WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1, 2
""").df()

import pandas as pd

trap = features_h1.merge(forward, on=["client_hash_id", "content_hash_id"], how="inner")
trap = trap[trap["imp_h1"] > 0].copy()  # ratio undefined at imp_h1 == 0
trap["is_declining"] = (trap["imp_h2"] < 0.8 * trap["imp_h1"]).astype(int)

print(f"{len(trap):,} rows with a defined label")
print(trap["is_declining"].value_counts(normalize=True).rename("share"))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

151,980 rows with a defined label
is_declining
0    0.673168
1    0.326832
Name: share, dtype: float64


In [10]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, balanced_accuracy_score

honest_features = ["imp_h1", "clicks_h1", "avg_position_h1", "active_days_h1", "ga4_covered_days_h1"]
X_honest = trap[honest_features].fillna(0)
y = trap["is_declining"]

majority_baseline = max(y.mean(), 1 - y.mean())
print(f"Majority-class baseline (always predict the bigger class): {majority_baseline:.3f}")
print("Using class_weight='balanced' below, so plain accuracy isn't directly comparable to that")
print("baseline -- balanced accuracy is the fair comparison (see w02's accuracy-vs-imbalance point).\n")

honest_model = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_honest, y)
honest_acc = accuracy_score(y, honest_model.predict(X_honest))
honest_bal_acc = balanced_accuracy_score(y, honest_model.predict(X_honest))
print(f"Honest quick score (5 features, all from before the decision cutoff):")
print(f"  accuracy:          {honest_acc:.3f}")
print(f"  balanced accuracy: {honest_bal_acc:.3f}  <- the fair number to trust here")

# Now the trap: add imp_h2 -- the label's own generating column -- as a 6th "feature".
leaky_features = honest_features + ["imp_h2"]
X_leaky = trap[leaky_features].fillna(0)

leaky_model = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_acc = accuracy_score(y, leaky_model.predict(X_leaky))
leaky_bal_acc = balanced_accuracy_score(y, leaky_model.predict(X_leaky))
print(f"\n'Leaky' quick score (same 5 features + imp_h2):")
print(f"  accuracy:          {leaky_acc:.3f}")
print(f"  balanced accuracy: {leaky_bal_acc:.3f}  <- jumps toward perfect either way")
print()
print(export_text(leaky_model, feature_names=leaky_features))

Majority-class baseline (always predict the bigger class): 0.673
Using class_weight='balanced' below, so plain accuracy isn't directly comparable to that
baseline -- balanced accuracy is the fair comparison (see w02's accuracy-vs-imbalance point).

Honest quick score (5 features, all from before the decision cutoff):
  accuracy:          0.552
  balanced accuracy: 0.581  <- the fair number to trust here

'Leaky' quick score (same 5 features + imp_h2):
  accuracy:          0.787
  balanced accuracy: 0.710  <- jumps toward perfect either way

|--- imp_h2 <= 3.50
|   |--- imp_h2 <= 0.50
|   |   |--- active_days_h1 <= 1.50
|   |   |   |--- class: 1
|   |   |--- active_days_h1 >  1.50
|   |   |   |--- class: 1
|   |--- imp_h2 >  0.50
|   |   |--- imp_h1 <= 1.50
|   |   |   |--- class: 0
|   |   |--- imp_h1 >  1.50
|   |   |   |--- class: 1
|--- imp_h2 >  3.50
|   |--- active_days_h1 <= 5.50
|   |   |--- imp_h2 <= 7.50
|   |   |   |--- class: 0
|   |   |--- imp_h2 >  7.50
|   |   |   |--- cl

The tree splits almost entirely on `imp_h2`, because `imp_h2` *is* the quantity the label is a
threshold rule on top of — the model isn't finding a pattern, it's reading the answer key. That's
the same lesson notebook 02 taught with `trend_pct`, reproduced here on real warehouse data by
me, not inherited from the starter pipeline.

Deleting the leak and keeping the honest number:

In [11]:
# Delete the leak, keep the honest number.
del X_leaky, leaky_model
print(f"Honest quick score, kept: balanced accuracy {honest_bal_acc:.3f} (raw accuracy {honest_acc:.3f})")
print(f"Leaky quick score, discarded: balanced accuracy {leaky_bal_acc:.3f} (raw accuracy {leaky_acc:.3f})")
print("(included the label's own generating column, imp_h2)")

Honest quick score, kept: balanced accuracy 0.581 (raw accuracy 0.552)
Leaky quick score, discarded: balanced accuracy 0.710 (raw accuracy 0.787)
(included the label's own generating column, imp_h2)


## 4. One named limitation of this slice

I built this contract without `dim_content`. The lane guide names `dim_content` as a real table
(519,606 rows, one per content item) and tells me it holds content metadata and join context, but
it doesn't give me a confirmed column list the way it does for `dim_clients` and
`fact_content_daily_performance` — and I don't have a way to browse the dataset's schema directly
from here. Rather than guess column names, I left it out entirely. That means this contract has
zero content-type, word-count, or query-context signal — every feature above is pure time-series
behavior. That's a real gap, not a stylistic choice: two pages with identical `imp_h1` /
`avg_position_h1` could be completely different kinds of content, and I currently can't tell them
apart. Before I build the real feature set for modeling weeks, I need to actually inspect
`dim_content`'s schema (a `DESCRIBE` / `LIMIT 5` against it, the same way I confirmed
`fact_march`'s columns in section 0) rather than carry this gap forward silently.

## 5. Self-check

The part of this notebook I trust least is the label. Splitting March into two 15/16-day halves
and calling a >20% drop "declining" is a direct, deliberate compression of the real design I keep
pointing at (prior 90 days → next 30 days) down to something that fits inside one partition and
one sitting — useful for proving the query mechanics and the leak honestly, but the 16-day
forward window is short enough that ordinary week-to-week noise could easily produce a false
"decline" that has nothing to do with the kind of sustained drop `docs/ml-intern-dataset-and-lane-guide.md`
distinguishes from real decline (section 7: consolidation, seasonality, noise). I don't think
that disqualifies this notebook — the job this week was proving I can state and verify a contract,
not ship a label I'd defend at the capstone — but I shouldn't let a clean-looking accuracy number
here convince me the label itself is ready.

The grain check in 3a was worth doing explicitly rather than assuming the guide's documented key
was actually enforced in this partition — a partition-write bug or a bad upstream join could
silently duplicate rows and I'd have built every feature on top of that without ever seeing it in
the feature frame itself.

Leaving `dim_content` out was the right call for this week rather than guessing at column names
and shipping a notebook that might silently reference something that doesn't exist, but it's a
gap I'm carrying forward on purpose, named in section 4, not one I'm pretending isn't there.

Going into the modeling weeks, the concrete thing I want to test is whether the real design (prior
90 days of features, a next-30-days label, evaluated under a client-holdout split like
`scripts/03_train_model.py` uses) holds up anywhere near as well as this compressed mid-month
version suggests — and whether `dim_content`, once I've actually looked at its schema, adds
enough over pure time-series signal to be worth the join complexity.

**What the real run actually showed, and one mistake I caught before submitting.** GA4 coverage
came in at 4.2% of March's rows — lower than I expected going in, and a concrete number behind
the "unbalanced panel, GSC-only early history" warning in the lane guide rather than an abstract
caveat. The join between the feature window and the forward window also dropped rows from
319,759 to 151,980 — content items that only have activity in one half of the month fall out of
the labeled set entirely, which I hadn't thought through until I saw the number; it means my
151,980-row label set is implicitly filtered toward pages active across the whole month, not a
clean random slice of March.

The more important catch: my first version of the trap compared the honest model's raw accuracy
(0.552) directly against the majority-class baseline (0.673) and made the honest features look
worse than guessing — but the model uses `class_weight="balanced"`, which trades raw accuracy for
per-class fairness on purpose, so that comparison wasn't apples to apples. Balanced accuracy
(0.581 honest vs. 0.710 leaky) is the number I should trust, and it's the same accuracy-can-mislead
lesson `w02` already made me argue for Precision@50 over accuracy — I built a cell that walked
into the exact trap I already knew about, on data I was less familiar with. Worth remembering
that "I've made this argument before" doesn't mean I won't repeat the mistake in a new context.